# Level 2a — From cells to cell types

### CAJAL Neuromics 2026 · Project 15: Mapping the Spatial Cellular Architecture of the Brain

**~ ½ day · graded**

> *"We have a molecular parts-list of the developing cortex. Which of those cell types can we
> recognise, cell-by-cell, in the tissue itself?"*

### 📖 The reveal

Levels 0–1 kept the dataset anonymous. Here it is: this is the **developing human neocortex**,
from **Wang et al., *Nature* 2025** (*"Molecular and cellular dynamics of the developing human
neocortex"*, Kriegstein lab). The authors built a paired single-nucleus **multiome** atlas (RNA +
ATAC) spanning first trimester → adolescence in two areas — prefrontal cortex (**PFC**) and primary
visual cortex (**V1**) — and, on a subset, ran **MERFISH** spatial transcriptomics with a 300-gene
panel. Level 2 reproduces the heart of their spatial analysis (their Fig. 2). Over 2a + 2b you will
go from raw spatial cells → cell types → tissue architecture → cell–cell communication.

**This notebook (2a)** does the first step: assign a **cell-type identity** to each of the 404,030
MERFISH cells, and *earn the right to trust those labels* — because everything in 2b is built on them.

**Learning objectives — by the end you can:**
- Explain why a 300-gene spatial panel is annotated by **reference mapping** from a matched
  single-cell atlas, rather than by clustering it from scratch.
- Run **CellMapper** (a CCA joint embedding + kNN label transfer) to move cell-type labels from the
  multiome RNA reference onto the spatial cells.
- **Validate** a transferred annotation three ways: canonical markers, held-out-gene imputation, and
  agreement with an *independent* labelling.
- Judge annotation quality honestly — where it is trustworthy (lineage/class) and where it is not
  (fine subtypes).

**How to work through this notebook**
- 🔬 **TASK** cells are for you to complete. 💡 **HINT** points the way, ❓ **QUESTION** is for
  reflection, ⚠️ **CHECKPOINT** tells you what to expect so you know you're on track.
- **Kernel:** use **`Spatial Brain (SIF)`** (top-right in JupyterLab).
- Yellow **`FutureWarning`** boxes are normal — these libraries evolve fast; they are *not* errors.
- Some cells run for **a few minutes** on 404k cells (loading the reference, the CellMapper steps) —
  a slow cell is working, not hung.
- **Data — read/write map (you all run this in parallel, so this matters):** the cohort and the
  reference are **read-only** on the shared project filesystem; everything you *write* (the annotated
  object, figures) goes into **your own repo's `data/` and `figures/`**. Never write to the shared
  paths. The reference is ~3 GB — loading it takes a minute or two; that's normal.


---
## 0. Setup

In [ ]:
import numpy as np
import pandas as pd

import scanpy as sc
import squidpy as sq
from cellmapper import CellMapper

sc.settings.verbosity = 1
sc.set_figure_params(dpi=80, frameon=False, figsize=(4, 4))
%matplotlib inline

# Shared, READ-ONLY project data
DATA = "/shared/projects/tp_2630_ubordeaux_neuromics_184418/projects/C15/data"
COHORT_PATH = f"{DATA}/wang2025_merfish/processed/wang2025_merfish_cells_student.h5ad"
REF_PATH = f"{DATA}/wang2025_multiome/processed/wang2025_multiome_rna.h5ad"
GROUND_TRUTH_PATH = f"{DATA}/wang2025_merfish/processed/wang2025_merfish_cells.h5ad"  # authors' labels — used ONLY for the final comparison

# YOUR outputs -> your repo's data/ (git-ignored, per-student)
from spatialbrain import FilePaths

OUT = FilePaths.dataset("wang2025_merfish").create()
print("writing outputs under:", OUT.processed)

---
## 1. The spatial cohort

The MERFISH cohort is **six sections** (3 PFC + 3 V1, spanning second trimester → infancy),
**404,030 cells × 300 genes**. `X` arrives pre-filled with a normalised matrix (we overwrite it from
raw below); `layers['counts']` holds the raw counts; `obsm['spatial']` the physical (x, y) coordinate
of every cell; and `.obs` carries the experimental design (`Sample_ID`, `Region`, age) plus Vizgen QC.
The authors' cell-type and niche columns have been **removed** — that's your job to reconstruct.

In [ ]:
cohort = sc.read_h5ad(COHORT_PATH)
cohort

🔬 **TASK 1.1 — Get oriented.** Using `cohort.obs`, print (1) how many cells per `Sample_ID`, and
(2) the cross-tabulation of `Region` (PFC/V1) against age
(`Estimated_postconceptional_age_in_days`). How many samples, regions and ages are there?

💡 **HINT:** `.value_counts()` on a `.obs` column tallies cells per category; `pd.crosstab` cross-tabulates two `.obs` columns against each other.

In [ ]:
# YOUR CODE HERE — see the hint above
...

⚠️ **CHECKPOINT.** Six `Sample_ID`s, two `Region`s (PFC, V1), three age groups. The section sizes
range from ~30k to ~110k cells.

---
## 2. Quality control — already done, but look anyway

These 404,030 cells are the authors' **post-QC** set. Their thresholds (from the paper's methods):
cell **volume > 10 µm³**, **25 ≤ nCount_Vizgen ≤ 2000** transcripts, **nFeature_Vizgen > 10** genes.
You won't re-filter, but always *look* at the QC distributions of data you're handed.

In [ ]:
sc.pl.violin(cohort, ["nCount_Vizgen", "nFeature_Vizgen", "volume"], jitter=0, multi_panel=True)

❓ **QUESTION.** The panel is 300 genes and counts are capped at 2000. Compared with a
whole-transcriptome scRNA-seq dataset (thousands of genes per cell), what does this tell you about
how much information you have per cell to *cluster* cell types de novo — and why reference mapping
might be the better move here?

---
## 3. Normalise, and look at the batch structure

Standard log-normalisation from raw counts (total-count normalise to 10,000, then `log1p`), then a
quick PCA/UMAP **on a subsample** (404k cells is slow to embed and we only want a look) coloured by
sample. We keep the subsample `sub` — we reuse it later to *see* our labels on an embedding.

In [ ]:
cohort.X = cohort.layers["counts"].copy()
sc.pp.normalize_total(cohort, target_sum=1e4)
sc.pp.log1p(cohort)

# quick embedding on a 40k subsample, just to see batch structure (reused later for label views)
sub = sc.pp.subsample(cohort, n_obs=40000, copy=True, random_state=0)
sc.pp.pca(sub, n_comps=50)
sc.pp.neighbors(sub, n_neighbors=15)
sc.tl.umap(sub)
sc.pl.umap(sub, color=["Sample_ID", "Region"], wspace=0.4)

❓ **QUESTION.** Do cells group mostly by biology or mostly by `Sample_ID`? If sections separate in
the embedding, de-novo clustering would partly chase batch, not cell type. Reference mapping sidesteps
this: every cell is compared to the *same* reference in a shared space. That's the approach we take next.

---
## 4. The cohort in space — look before you compute

This is spatial data: every cell has a physical location, and *that is the whole point*. Before we
put a single label on anything, **look at all six sections in tissue space**. First the transcript
count per cell (a spatial QC view — folds, edges and low-coverage patches show up here), then a handful
of **canonical lineage markers** on one section, which already trace the developing cortex's radial
architecture (progenitors on the ventricular side, neurons stacked outward).

In [ ]:
cohort.obs["Sample_ID"] = cohort.obs["Sample_ID"].astype("category")
samples = cohort.obs["Sample_ID"].cat.categories.tolist()

# all six sections in space, coloured by transcript count (spatial QC)
sq.pl.spatial_scatter(
    cohort,
    library_key="Sample_ID",
    library_id=samples,
    color="nCount_Vizgen",
    shape=None,
    size=3,
    ncols=3,
    cmap="viridis",
    title=samples,
    figsize=(13, 8),
)

🔬 **TASK 4.1 — Preview the architecture with marker genes.** On **one** section, plot a few
canonical lineage markers in space: a progenitor (`VIM`), an excitatory-neuron (`SATB2`), an
inhibitory (`GAD1`), an astrocyte (`AQP4`) and an oligodendrocyte (`OLIG1`) gene.

💡 **HINT:** `sq.pl.spatial_scatter` with `library_id=<one Sample_ID>` and `color=<list of genes>`
draws one panel per gene. Keep only genes present in `cohort.var_names`. Turn `shape=None` and use a
small `size` so it renders fast.

In [ ]:
# YOUR CODE HERE — see the hint above
...

❓ **QUESTION.** Even without cell-type labels, can you already see *layering* — one marker on the
outer/ventricular edge, another band further out? That radial gradient is the developing cortex; the
rest of this notebook makes it quantitative by putting a cell type on every dot.

---
## 5. The annotation reference

We annotate by transferring labels from the **multiome RNA modality** — the same nuclei, fully
sequenced (all genes), with the authors' cell-type labels. (Data-wrangling note: in the original MuData
the labels sit on the **global** `.obs`, not on `mdata['RNA'].obs`; we pre-extracted the RNA modality
with labels attached and genes filtered to those expressed in ≥ 10 nuclei.)

**One mapping, two jobs.** The mapping we build in §6 will do double duty: transfer labels *now* (2a)
and, in 2b, **impute** the genes we need for cell–cell communication. Most ligand–receptor genes are
**off-panel** (e.g. `SST` is measured but its receptors are not), so we keep on the reference the 300
panel genes **plus** the ligand–receptor genes 2b will need. The joint embedding still only uses the
300 shared panel genes — the extra genes just ride along so the same mapping can impute them later.

In [ ]:
ref = sc.read_h5ad(REF_PATH)  # ~3 GB; takes a minute
ref.obs["type"] = ref.obs["type"].astype("category")
ref.obs["class"] = ref.obs["class"].astype("category")
print(ref)
print("\ncell types in reference:", ref.obs["type"].cat.categories.size)
print("panel genes present in reference:", cohort.var_names.isin(ref.var_names).sum(), "/", cohort.n_vars)

# panel genes + the ligand-receptor genes 2b needs (from the LIANA consensus resource)
import liana as li

lr = set()
for col in ("ligand", "receptor"):
    for g in li.resource.select_resource("consensus")[col]:
        lr.update(str(g).split("_"))
lr_in_ref = sorted(lr & set(ref.var_names))
impute_genes = sorted(set(cohort.var_names) | set(lr_in_ref))
ref_imp = ref[:, impute_genes].copy()
print(f"ligand/receptor genes carried for 2b: {len(lr_in_ref)}  ->  reference for mapping: {ref_imp.n_vars} genes")

---
## 6. Annotate by reference mapping (CellMapper)

**CellMapper** builds a **joint embedding** of query (spatial) and reference (RNA) cells with **fast
CCA** (on the shared 300 genes), finds each spatial cell's nearest reference neighbours, and transfers
labels across that kNN graph. On CPU we use `pynndescent` for the neighbour search.

The authors annotated their spatial cells **independently** of any reference (clustering + markers).
We use an *orthogonal* method (mapping from the molecular atlas) — so the comparison at the end is
genuine cross-validation, not circular reasoning.

🔬 **TASK 6.1 — Transfer the labels.** Build a `CellMapper(cohort, ref_imp)`, compute the fast-CCA
joint embedding, then `map` **both** the `"type"` and `"class"` columns onto the cohort in one call.

💡 **HINT:** three-step API — construct, `compute_fast_cca(n_comps=50)`, then
`.map(obs_keys=["type", "class"], use_rep="X_cca", ...)`. Pass **`only_yx=True`** (build just the
query→reference kNN, not all four — much faster on 404k cells) and `knn_method="pynndescent"` (CPU).
Afterwards `cohort.obs` gains `type_pred`/`type_conf` and `class_pred`/`class_conf`.

In [ ]:
# YOUR CODE HERE — see the hint above
...

❓ **QUESTION.** `type_conf` is a per-cell confidence. Where do you expect it to be *low* — and does
that line up with the cell types a 300-gene panel struggles to tell apart? (We'll test the labels three
ways below before trusting them.)

---
## 7. Validation I — do canonical markers land on the right types?

The cleanest sanity check uses genes **actually measured** in the panel (not imputed — that would be
circular). A dotplot of lineage markers grouped by `type_pred` should show each marker lighting up in
the expected cell type.

In [ ]:
markers = {
    "RG/progenitor": ["VIM", "HES1", "HOPX", "TFAP2C", "CRYAB"],
    "IPC-EN": ["EOMES", "HES6"],
    "Excitatory": ["SATB2", "NEUROD1", "CUX2", "RORB", "BCL11B", "FEZF2"],
    "Inhibitory": ["GAD1", "GAD2", "DLX2", "SST", "VIP", "SP8"],
    "Astrocyte": ["AQP4", "GJA1", "SLC1A2"],
    "OPC/Oligo": ["OLIG1", "PDGFRA", "PLP1"],
    "Vascular": ["CLDN5", "FLT1"],
}
markers = {k: [g for g in v if g in cohort.var_names] for k, v in markers.items()}
sc.pl.dotplot(cohort, markers, groupby="type_pred", standard_scale="var", figsize=(14, 8))

🔬 **TASK 7.1 — Read the dotplot.** Name two predicted types whose marker pattern clearly matches
their expected identity (e.g. an excitatory type expressing `SATB2`, an inhibitory type expressing
`GAD1`/`GAD2`). Are there any types whose markers look ambiguous?

---
## 8. Validation II — held-out-gene imputation

A stricter test: **hide** some panel genes from the joint embedding, ask the mapping to **predict**
their expression from the reference, and correlate predicted vs measured. If the neighbours are
biologically right, held-out genes are recoverable. CellMapper does the heavy lifting: mask genes with
a `is_train` column (`mask_var`), impute with `map_layers`, and score with **`evaluate_expression_transfer`**
(per-gene, per-section Pearson r stored back on the object). We pass **`target_libsize`** so imputed
counts are scaled to each cell's measured depth over the *training* genes.

In [ ]:
rng = np.random.default_rng(0)
is_test = np.zeros(cohort.n_vars, bool)
is_test[rng.choice(cohort.n_vars, size=60, replace=False)] = True
cohort.var["is_train"] = ~is_test
cohort.var["is_test"] = is_test

# a second mapping that never sees the held-out genes, then impute them back and score
ref_panel = ref[:, cohort.var_names].copy()
ref_panel.var["is_train"] = ~is_test

cm_val = CellMapper(cohort, ref_panel)
cm_val.compute_fast_cca(n_comps=50, mask_var="is_train")  # embedding from training genes only
lib_train = np.asarray(cohort[:, cohort.var["is_train"].values].layers["counts"].sum(1)).ravel()
cm_val.map(
    layer_key="counts",
    use_rep="X_cca",
    n_neighbors=30,
    knn_method="pynndescent",
    only_yx=True,
    target_libsize=lib_train,
)

cm_val.evaluate_expression_transfer(
    layer_key="counts", groupby="Sample_ID", comparison_method="pearson", test_var_key="is_test"
)
print(cm_val.expression_transfer_metrics)

🔬 **TASK 8.1 — See where imputation works and where it fails.** The per-gene correlations now
live in `cohort.var["metric_pearson"]`. Print the median over held-out genes and a histogram, then pick
the **2 best** and **2 worst** held-out genes and plot **measured vs imputed** for them **in space** on
one section. High-r genes should reproduce the spatial pattern; low-r genes won't.

💡 **HINT:** sort `cohort.var.loc[cohort.var["is_test"], "metric_pearson"]`; the imputed matrix is
`cm_val.query_imputed` (same cells, aligned coordinates). Plot measured (`cohort`) and imputed
(`cm_val.query_imputed`) for the same section with `sq.pl.spatial_scatter`.

In [ ]:
# YOUR CODE HERE — see the hint above
...

In [ ]:
# YOUR CODE HERE — see the hint above
...

⚠️ **CHECKPOINT.** The median per-gene correlation is modest (around **0.3–0.5**) with a spread.
That is the honest reality of imputing from ~240 genes: the mapping captures broad lineage-level
expression, not every gene's fine detail. The spatial plots make it concrete — a high-r gene's imputed
map mirrors the measured one; a low-r gene's does not.

❓ **QUESTION.** Why does validating with measured markers (§7) and held-out genes (§8) test the
annotation *without ever looking at the authors' labels* — and why does that make it a fair test?

---
## 9. Validation III — an external opinion (CellTypist)

CellMapper used *this paper's own* reference. A useful cross-check is a **pretrained, external**
classifier that never saw this dataset. We run two CellTypist models and read them critically — both
have a **stage/region gap** with our second-trimester→infancy neocortex: `Developing_Human_Brain`
(first-trimester) and `Adult_Human_PrefrontalCortex` (adult).

🔬 **TASK 9.1 — Run the two baselines and *look at the labels*.** Annotate the cohort with each
model (`majority_voting=False`), store each into its own `obs` column, then colour the §3 embedding
`sub` by the two external label sets. Abstract label lists mean little; seeing them on the embedding
does.

💡 **HINT:** `celltypist.annotate(cohort, model=m).predicted_labels["predicted_labels"]` gives the
per-cell calls. CellTypist expects log1p-CP10k — how we normalised. Copy each `obs` column onto `sub`
via `sub.obs_names` and `sc.pl.umap`.

In [ ]:
# YOUR CODE HERE — see the hint above
...

In [ ]:
# YOUR CODE HERE — see the hint above
...

❓ **QUESTION.** The external models return labels like *"Hindbrain radial glia"* or *"Oligo MOG
OPALIN"* — plausible cell classes but from the wrong stage/region. What does this tell you about the
value of a **stage-matched, same-study reference** (our multiome) over a generic pretrained one?

---
## 10. The orthogonal comparison — how did we do?

Now we allow ourselves the authors' labels (in the original file), purely to *evaluate*. They were made
**independently** (clustering + markers), so agreement is real cross-validation. CellMapper scores label
transfer for us (`evaluate_label_transfer`) and draws the confusion matrix (`plot_confusion_matrix`).

⚠️ **A taxonomy caveat.** The multiome reference carries **34 types / 6 classes** (incl. an `Unknown`
bucket and extra progenitor/immature types); the MERFISH ground truth resolves **29 types / 5 classes**.
So the mapper can emit labels that simply *cannot* exist in the ground truth — real granularity
mismatches, not errors. Expect strong agreement at **class** level, weaker at fine **type** level.

In [ ]:
truth = sc.read_h5ad(GROUND_TRUTH_PATH, backed="r")
cohort.obs["type"] = pd.Categorical(truth.obs.loc[cohort.obs_names, "type"].values)  # authors' labels
cohort.obs["class"] = pd.Categorical(truth.obs.loc[cohort.obs_names, "class"].values)

# built-in scoring of our reference-mapped labels vs the authors' independent labels
cm.evaluate_label_transfer("class")
print("CLASS:", {k: round(v, 3) for k, v in cm.label_transfer_metrics.items()})
cm.evaluate_label_transfer("type")
print("TYPE :", {k: round(v, 3) for k, v in cm.label_transfer_metrics.items()})

In [ ]:
cm.plot_confusion_matrix(
    pred_key="class",
    normalize="true",
    include_values=True,
    figsize=(7, 6),
    title="class — authors (rows) vs ours (cols)",
)

🔬 **TASK 10.1 — Map the agreement, don't just score it.** Add a per-cell `class_agree`
(`class_pred` == authors' `class`) and show *where* we agree and diverge — in **space** (two sections)
and on the **embedding** (`sub`, next to the authors' class and ours). Disagreements should concentrate
in specific lineages/regions, not scatter at random.

💡 **HINT:** build the boolean, cast to category, `sq.pl.spatial_scatter(..., color="class_agree")`
for two `samples`, and copy `class`/`class_pred`/`class_agree` onto `sub` for `sc.pl.umap`.

In [ ]:
# YOUR CODE HERE — see the hint above
...

⚠️ **CHECKPOINT.** **Class-level agreement ≈ 0.85–0.90**, **type-level ≈ 0.5** — and that gap *is*
the lesson: two independent methods strongly agree on **lineage** (neuron? glia? progenitor?) but
diverge on fine **subtype**, where a 300-gene panel and different pipelines genuinely see things
differently. The agreement map should show disagreement concentrating in the hardest lineages, not
sprinkled at random.

🚀 **Going further (optional).** Cluster the *reference* on the panel genes only and find the type
pairs whose centroids sit closest. Does the spatial mapper confuse those same pairs (`type` vs
`type_pred`)? If so, that's an information limit of the panel, not a mapping bug.

---
## 11. Impute the signalling genes, and save for 2b

We validated imputation in §8, so we now *use* it: apply the **same mapping from §6** to impute the
ligand–receptor genes (mostly off-panel) that 2b's cell–cell communication needs — no second mapping,
no recomputation. We save two objects to **your** repo `data/`: the annotated cohort (measured genes +
labels + coordinates) and the imputed ligand–receptor matrix. **2b reads both from here** — the
analysis builds up across notebooks.

In [ ]:
# same operator as the label transfer in §6 -> impute counts for all carried genes
lib_all = np.asarray(cohort.layers["counts"].sum(1)).ravel()
cm.map_layers(key="counts", target_libsize=lib_all)

# keep just the ligand-receptor genes 2b needs; lean obs; float32 to keep the file small
lr_imputed = cm.query_imputed[:, lr_in_ref].copy()
lr_imputed.obs = cohort.obs[["type_pred", "Sample_ID", "Region"]].copy()
lr_imputed.X = lr_imputed.X.astype("float32")
print("imputed ligand/receptor object:", lr_imputed.shape)

In [ ]:
# drop the authors' truth columns we borrowed for scoring; keep our derived annotation
cohort.obs = cohort.obs.drop(columns=["type", "class"])
cohort_out = OUT.processed / "cohort_annotated.h5ad"
cohort.write_h5ad(cohort_out)
lr_out = OUT.processed / "cohort_lr_imputed.h5ad"
lr_imputed.write_h5ad(lr_out)
print("wrote", cohort_out.name, f"({cohort_out.stat().st_size / 1e6:.0f} MB)")
print("wrote", lr_out.name, f"({lr_out.stat().st_size / 1e6:.0f} MB)")

---
## Where this is heading

You've put a trustworthy cell-type label on every spatial cell — and you know *how far* to trust it
(lineage: yes; fine subtype: with caution) — and you've imputed the signalling genes 2b needs. In
**Level 2b** we stop asking *what* the cells are and start asking *where* they are and *how they talk*:
we place these types in physical space, find that they self-organise into the **cortical layers and
zones** (niches), and trace the **cell–cell communication** that helps build the cortex — reproducing
the core of Wang et al.'s Figure 2.

In [ ]:
import session_info2

session_info2.session_info(os=True, cpu=True)